# FinanceAI Dataset Builder (Improved)
## Team 38 (LATAM) · Oracle ONE G9 · Proyecto FinanceAI

**Improvements Made:**
- ✅ Transaction amounts scaled 50x for realistic financial profiles
- ✅ Adjusted scoring thresholds for better discrimination
- ✅ Added data quality validation and warnings
- ✅ Enhanced recommendations with category breakdown
- ✅ Comprehensive logging and statistics
- ✅ Better error handling with try/except blocks
- ✅ Intermediate DataFrame export for debugging

In [ ]:
import subprocess
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

"""
BUILD_DATASET_FINANCEAI.PY (IMPROVED)
=====================================
Team 38 (LATAM) · Oracle ONE G9 · Proyecto FinanceAI

Construye el dataset híbrido explícito (Propuesta 3):
  - Capa 1 (real):    transacciones reales anonimizadas -> entrena el clasificador de gastos
  - Capa 2 (sintética): perfil financiero por usuario -> calcula perfil_financiero + recomendaciones

USO:
    1. pip install -r requirements.txt
    2. Configurar credenciales de Kaggle (ver función descargar_datasets)
    3. python build_dataset_financeai_improved.py

SALIDA:
    data/processed/financeai_dataset_hibrido.csv
    data/processed/df_transacciones_debug.csv (opcional)
    data/processed/df_perfiles_debug.csv (opcional)
"""

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================
RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

N_USUARIOS = 500          # número de usuarios sintéticos a generar
RANDOM_SEED = 42
TRANSACTION_SCALE = 50    # Multiplicador para escalar transacciones (MEJORA: era 1x)
DEBUG_MODE = True         # Exportar DataFrames intermedios para debugging

np.random.seed(RANDOM_SEED)

# Las 8 categorías definidas para el proyecto FinanceAI
CATEGORIAS_PROYECTO = [
    "Alimentación", "Transporte", "Salud", "Vivienda",
    "Educación", "Ocio", "Servicios", "Otros",
]

# Mapeo de categorías originales del dataset de Kaggle -> categorías del proyecto.
MAPEO_CATEGORIAS = {
    "groceries": "Alimentación",
    "restaurant": "Alimentación",
    "food": "Alimentación",
    "supermarket": "Alimentación",
    "transport": "Transporte",
    "fuel": "Transporte",
    "taxi": "Transporte",
    "public transport": "Transporte",
    "health": "Salud",
    "pharmacy": "Salud",
    "medical": "Salud",
    "rent": "Vivienda",
    "housing": "Vivienda",
    "mortgage": "Vivienda",
    "utilities": "Servicios",
    "bills": "Servicios",
    "internet": "Servicios",
    "education": "Educación",
    "tuition": "Educación",
    "books": "Educación",
    "entertainment": "Ocio",
    "streaming": "Ocio",
    "leisure": "Ocio",
    "shopping": "Otros",
    "other": "Otros",
}


def log_stage(stage_name: str, message: str = ""):
    """Función auxiliar para logging consistente."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {stage_name}: {message}")


# ============================================================
# PASO 1 · Descarga de los datasets fuente (Kaggle API)
# ============================================================
def descargar_datasets():
    """
    Requiere:
      pip install kaggle
      Un token de API de Kaggle en ~/.kaggle/kaggle.json
      (Kaggle > Account > Create New API Token)

    Descarga las dos fuentes usadas en la metodología híbrida:
      - Capa 1: transacciones reales anonimizadas
      - Capa 2: perfil financiero de referencia
    """
    log_stage("PASO 1", "Descargando datasets de Kaggle...")
    
    datasets = [
        "artemkabseu/financial-transactions-dataset-expenses-and-income",  # Capa 1
        "miadul/personal-finance-ml-dataset",                              # Capa 2
    ]
    
    for ds in datasets:
        try:
            log_stage("PASO 1", f"Descargando {ds}...")
            subprocess.run(
                ["kaggle", "datasets", "download", "-d", ds, "-p", str(RAW_DIR), "--unzip"],
                check=True,
            )
            log_stage("PASO 1", f"✓ {ds} descargado exitosamente")
        except subprocess.CalledProcessError as e:
            log_stage("PASO 1", f"⚠️ Error al descargar {ds}: {e}")
            raise
        except FileNotFoundError:
            log_stage("PASO 1", "⚠️ Kaggle CLI no encontrado. Instala con: pip install kaggle")
            raise


# ============================================================
# PASO 2 · Cargar y normalizar Capa 1 (transacciones reales)
# ============================================================
def cargar_capa1_transacciones(path_csv: Path) -> pd.DataFrame:
    """
    Carga el CSV de transacciones y lo deja en el esquema estándar del proyecto:
    fecha, descripcion, categoria, valor.
    
    MEJORA: Ahora escala las transacciones por TRANSACTION_SCALE para que sean realistas.
    """
    log_stage("PASO 2", f"Cargando transacciones de {path_csv.name}...")
    
    df = pd.read_csv(path_csv)
    log_stage("PASO 2", f"Columnas originales: {df.columns.tolist()}")
    log_stage("PASO 2", f"Total de transacciones cargadas: {len(df)}")
    
    df = df.rename(columns={
        "date_time": "fecha",
        "tags": "descripcion",
        "category": "categoria_original",
        "amount": "valor",
    })

    columnas_requeridas = {"fecha", "descripcion", "categoria_original", "valor"}
    faltantes = columnas_requeridas - set(df.columns)
    if faltantes:
        raise ValueError(
            f"Faltan columnas esperadas tras el rename: {faltantes}. "
            "Revisa los nombres reales del CSV descargado y ajusta el diccionario de 'rename'."
        )

    df["categoria"] = (
        df["categoria_original"].astype(str).str.strip().str.lower().map(MAPEO_CATEGORIAS)
    )
    df["categoria"] = df["categoria"].fillna("Otros")

    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce").abs()
    
    # MEJORA: Escalar transacciones para obtener perfiles más realistas
    df["valor"] = df["valor"] * TRANSACTION_SCALE
    
    df = df.dropna(subset=["valor"])

    log_stage("PASO 2", f"Transacciones después de limpiar: {len(df)}")
    log_stage("PASO 2", f"Valor promedio por transacción: ${df['valor'].mean():.2f}")
    log_stage("PASO 2", f"Valor máximo: ${df['valor'].max():.2f}")
    
    return df[["fecha", "descripcion", "categoria", "valor"]].reset_index(drop=True)


# ============================================================
# PASO 3 · Generar usuario_id sintético y distribuir transacciones
# ============================================================
def asignar_usuarios(df_transacciones: pd.DataFrame, n_usuarios: int = N_USUARIOS):
    """
    Genera IDs de usuario sintéticos y distribuye las transacciones reales entre ellos.
    La distribución no es uniforme (algunos usuarios "gastan más" que otros), usando
    una distribución de Dirichlet para simular variabilidad realista.
    """
    log_stage("PASO 3", f"Generando {n_usuarios} usuarios sintéticos...")
    
    usuario_ids = [f"user_{i:04d}" for i in range(n_usuarios)]

    pesos = np.random.dirichlet(np.ones(n_usuarios) * 2)
    df = df_transacciones.copy()
    df["usuario_id"] = np.random.choice(usuario_ids, size=len(df), p=pesos)

    transacciones_por_usuario = df.groupby("usuario_id").size()
    log_stage("PASO 3", f"Transacciones asignadas por usuario (promedio): {transacciones_por_usuario.mean():.2f}")
    log_stage("PASO 3", f"Min: {transacciones_por_usuario.min()}, Max: {transacciones_por_usuario.max()}")
    
    return df, usuario_ids


# ============================================================
# PASO 4 · Cargar Capa 2 (perfil financiero) como referencia estadística
# ============================================================
def cargar_capa2_referencia(path_csv: Path) -> pd.DataFrame:
    """
    Carga el dataset de perfil financiero público SOLO como referencia estadística.
    """
    log_stage("PASO 4", f"Cargando perfil de referencia de {path_csv.name}...")
    
    df = pd.read_csv(path_csv)
    log_stage("PASO 4", f"Registros de referencia cargados: {len(df)}")

    df = df.rename(columns={
        "monthly_income_usd": "ingreso_mensual",
        "debt_to_income_ratio": "nivel_endeudamiento",
        "savings_to_income_ratio": "ratio_ahorro",
        "credit_score": "score_crediticio",
    })

    columnas_requeridas = {"ingreso_mensual", "nivel_endeudamiento", "ratio_ahorro", "score_crediticio"}
    faltantes = columnas_requeridas - set(df.columns)
    if faltantes:
        raise ValueError(
            f"Faltan columnas esperadas tras el rename: {faltantes}. "
            "Revisa los nombres reales del CSV descargado y ajusta el diccionario de 'rename'."
        )

    df_clean = df[list(columnas_requeridas)].dropna()
    log_stage("PASO 4", f"Registros después de limpiar NaN: {len(df_clean)}")
    log_stage("PASO 4", f"Ingreso promedio (referencia): ${df_clean['ingreso_mensual'].mean():.2f}")
    
    return df_clean


def generar_perfiles_sinteticos(usuario_ids, df_referencia: pd.DataFrame) -> pd.DataFrame:
    """
    Genera el perfil financiero sintético de cada usuario_id MUESTREANDO del dataset de
    referencia (con reemplazo) en vez de copiarlo directamente, y agregando ruido leve.
    """
    log_stage("PASO 4B", f"Generando perfiles sintéticos para {len(usuario_ids)} usuarios...")
    
    muestra = (
        df_referencia
        .sample(n=len(usuario_ids), replace=True, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )
    muestra["usuario_id"] = usuario_ids

    # Ruido leve (±10%) para que no sean copias exactas de la fuente de referencia
    ruido = np.random.uniform(0.9, 1.1, size=len(muestra))
    muestra["ingreso_mensual"] = (muestra["ingreso_mensual"] * ruido).round(2)
    muestra["nivel_endeudamiento"] = muestra["nivel_endeudamiento"].clip(0, 1)

    # frecuencia_ahorro categórica derivada del ratio de ahorro
    muestra["frecuencia_ahorro"] = pd.cut(
        muestra["ratio_ahorro"],
        bins=[-np.inf, 0.05, 0.15, np.inf],
        labels=["Baja", "Media", "Alta"],
    ).astype(str)

    log_stage("PASO 4B", f"Ingreso promedio (sintético): ${muestra['ingreso_mensual'].mean():.2f}")
    log_stage("PASO 4B", f"Endeudamiento promedio: {muestra['nivel_endeudamiento'].mean():.3f}")
    log_stage("PASO 4B", f"Distribución de frecuencia de ahorro:\n{muestra['frecuencia_ahorro'].value_counts()}")
    
    return muestra[[
        "usuario_id", "ingreso_mensual", "nivel_endeudamiento",
        "frecuencia_ahorro", "score_crediticio",
    ]]


# ============================================================
# PASO 5 · Función de scoring del perfil financiero (MEJORADA)
# ============================================================
def calcular_perfil_financiero(row) -> str:
    """
    REGLA DE NEGOCIO MEJORADA:
    Combina ratio gasto/ingreso, nivel de endeudamiento y frecuencia de ahorro
    en un puntaje de riesgo. THRESHOLDS AJUSTADOS para mejor discriminación.
    """
    riesgo = 0

    # MEJORA: Thresholds más realistas (antes: 0.9 y 0.7)
    if row["ratio_gasto_ingreso"] > 0.6:
        riesgo += 2
    elif row["ratio_gasto_ingreso"] > 0.4:
        riesgo += 1

    # MEJORA: Thresholds más realistas (antes: 0.4 y 0.25)
    if row["nivel_endeudamiento"] > 0.5:
        riesgo += 2
    elif row["nivel_endeudamiento"] > 0.3:
        riesgo += 1

    if row["frecuencia_ahorro"] == "Baja":
        riesgo += 1
    elif row["frecuencia_ahorro"] == "Alta":
        riesgo -= 1

    # MEJORA: Añadimos categoría "En riesgo" con threshold más bajo
    if riesgo >= 4:
        return "En riesgo"
    if riesgo >= 2:
        return "En observación"
    return "Saludable"


def generar_recomendaciones(row) -> list:
    """
    MEJORA: Recomendaciones más específicas con breakdown de categorías.
    """
    recomendaciones = []

    # Advertencia si no hay transacciones
    if row["gasto_total"] == 0:
        recomendaciones.append("⚠️ Sin transacciones registradas en el período analizado")
        return recomendaciones
    
    # Identificar categoría principal
    categorias_gasto = {cat: row[cat] for cat in CATEGORIAS_PROYECTO if cat in row.index}
    top_category = max(categorias_gasto.items(), key=lambda x: x[1])
    
    if row["ratio_gasto_ingreso"] > 0.6:
        recomendaciones.append(
            f"🔴 CRÍTICO: Gastos muy altos. Reducir categoría '{top_category[0]}' (${top_category[1]:.2f})"
        )
    elif row["ratio_gasto_ingreso"] > 0.4:
        recomendaciones.append(
            f"🟡 MODERADO: Gastos moderados. Revisar categoría '{top_category[0]}' (${top_category[1]:.2f})"
        )
    
    if row["nivel_endeudamiento"] > 0.5:
        recomendaciones.append(
            "🔴 Nivel de endeudamiento crítico: Priorizar pago de deuda antes de nuevos compromisos"
        )
    elif row["nivel_endeudamiento"] > 0.3:
        recomendaciones.append(
            "🟡 Nivel de endeudamiento moderado: Monitorear deuda activamente"
        )
    
    if row["frecuencia_ahorro"] == "Baja":
        ahorro_recomendado = row["ingreso_mensual"] * 0.1
        recomendaciones.append(
            f"💰 Aumentar ahorro: Objetivo mínimo ${ahorro_recomendado:.2f} mensual (10% del ingreso)"
        )
    elif row["frecuencia_ahorro"] == "Alta":
        recomendaciones.append(
            "✅ Frecuencia de ahorro adecuada: Mantener este hábito positivo"
        )
    
    if not recomendaciones:
        recomendaciones.append("✅ Mantener los hábitos financieros actuales")

    return recomendaciones


# ============================================================
# PASO 6 · Unir Capa 1 + Capa 2 en el dataset final
# ============================================================
def construir_dataset_final(df_transacciones_usuario: pd.DataFrame, df_perfiles: pd.DataFrame) -> pd.DataFrame:
    log_stage("PASO 6", "Construyendo dataset final...")
    
    resumen_gastos = (
        df_transacciones_usuario
        .groupby(["usuario_id", "categoria"])["valor"]
        .sum()
        .unstack(fill_value=0)
        .reset_index()
    )

    # Asegurar que todas las categorías del proyecto existan como columnas
    for cat in CATEGORIAS_PROYECTO:
        if cat not in resumen_gastos.columns:
            resumen_gastos[cat] = 0.0

    df_final = df_perfiles.merge(resumen_gastos, on="usuario_id", how="left")
    df_final[CATEGORIAS_PROYECTO] = df_final[CATEGORIAS_PROYECTO].fillna(0.0)

    df_final["gasto_total"] = df_final[CATEGORIAS_PROYECTO].sum(axis=1)
    df_final["ratio_gasto_ingreso"] = (
        df_final["gasto_total"] / df_final["ingreso_mensual"].replace(0, np.nan)
    ).fillna(0)

    df_final["perfil_financiero"] = df_final.apply(calcular_perfil_financiero, axis=1)
    df_final["recomendaciones"] = df_final.apply(generar_recomendaciones, axis=1)

    log_stage("PASO 6", f"Dataset final: {len(df_final)} usuarios × {len(df_final.columns)} columnas")
    
    return df_final


# ============================================================
# VALIDACIÓN Y DIAGNOSTICO (NUEVA FUNCIÓN)
# ============================================================
def validar_dataset(df_final: pd.DataFrame) -> None:
    """
    MEJORA: Valida la calidad del dataset y emite advertencias si es necesario.
    """
    log_stage("VALIDACIÓN", "=" * 60)
    log_stage("VALIDACIÓN", "Ejecutando validación de calidad...")
    
    # Usuarios sin transacciones
    sin_transacciones = (df_final['gasto_total'] == 0).sum()
    porcentaje = (sin_transacciones / len(df_final)) * 100
    if sin_transacciones > 0:
        log_stage("VALIDACIÓN", f"⚠️ {sin_transacciones} usuarios ({porcentaje:.1f}%) sin transacciones")
    else:
        log_stage("VALIDACIÓN", f"✅ Todos los usuarios tienen transacciones registradas")
    
    # Estadísticas de ratio gasto/ingreso
    median_ratio = df_final['ratio_gasto_ingreso'].median()
    mean_ratio = df_final['ratio_gasto_ingreso'].mean()
    log_stage("VALIDACIÓN", f"Ratio gasto/ingreso - Media: {mean_ratio:.4f}, Mediana: {median_ratio:.4f}")
    
    if median_ratio < 0.1:
        log_stage("VALIDACIÓN", "⚠️ Advertencia: Los gastos son muy bajos respecto al ingreso")
        log_stage("VALIDACIÓN", "   Considera ajustar TRANSACTION_SCALE si los datos son irrealistas")
    elif median_ratio > 1.0:
        log_stage("VALIDACIÓN", "⚠️ Advertencia: Los gastos superan el ingreso (datos pueden ser inconsistentes)")
    else:
        log_stage("VALIDACIÓN", "✅ Ratio gasto/ingreso en rango realista")
    
    # Distribución de perfiles
    log_stage("VALIDACIÓN", "Distribución de perfiles financieros:")
    for profile, count in df_final['perfil_financiero'].value_counts().items():
        pct = (count / len(df_final)) * 100
        log_stage("VALIDACIÓN", f"  {profile}: {count} usuarios ({pct:.1f}%)")
    
    # Score crediticio vs ratio gasto/ingreso
    correlation = df_final['score_crediticio'].corr(df_final['ratio_gasto_ingreso'])
    log_stage("VALIDACIÓN", f"Correlación (score_crediticio ↔ ratio_gasto_ingreso): {correlation:.4f}")
    
    if abs(correlation) < 0.1:
        log_stage("VALIDACIÓN", "ℹ️ Nota: Baja correlación entre score y gastos (esperado en datos sintéticos)")
    
    # Detalles de ingresos por perfil
    log_stage("VALIDACIÓN", "Ingreso promedio por perfil:")
    for profile in df_final['perfil_financiero'].unique():
        avg_income = df_final[df_final['perfil_financiero'] == profile]['ingreso_mensual'].mean()
        log_stage("VALIDACIÓN", f"  {profile}: ${avg_income:.2f}")
    
    log_stage("VALIDACIÓN", "=" * 60)


# ============================================================
# MAIN
# ============================================================
def main():
    print("\n" + "="*70)
    print("FINANCEAI DATASET BUILDER - VERSIÓN MEJORADA")
    print("="*70 + "\n")
    
    try:
        # Descomentar si aún no se han descargado los CSV:
        # descargar_datasets()
        
        ruta_capa1 = RAW_DIR / "Expenses_clean.csv"
        ruta_capa2 = RAW_DIR / "synthetic_personal_finance_dataset.csv"

        # PASO 1-2: Cargar Capa 1
        df_transacciones = cargar_capa1_transacciones(ruta_capa1)

        # PASO 3: Asignar usuarios
        df_transacciones_usuario, usuario_ids = asignar_usuarios(df_transacciones)

        # PASO 4: Cargar y generar Capa 2
        df_referencia = cargar_capa2_referencia(ruta_capa2)
        df_perfiles = generar_perfiles_sinteticos(usuario_ids, df_referencia)

        # PASO 6: Construir dataset final
        df_final = construir_dataset_final(df_transacciones_usuario, df_perfiles)
        
        # VALIDACIÓN: Ejecutar checks de calidad
        validar_dataset(df_final)
        
        # Guardar dataset final
        salida = PROCESSED_DIR / "financeai_dataset_hibrido.csv"
        df_final.to_csv(salida, index=False)
        log_stage("SALIDA", f"✅ Dataset final guardado en {salida}")
        
        # OPCIONAL: Guardar DataFrames intermedios para debugging
        if DEBUG_MODE:
            debug_transacciones = PROCESSED_DIR / "df_transacciones_debug.csv"
            debug_perfiles = PROCESSED_DIR / "df_perfiles_debug.csv"
            df_transacciones_usuario.to_csv(debug_transacciones, index=False)
            df_perfiles.to_csv(debug_perfiles, index=False)
            log_stage("DEBUG", f"✅ Archivos de debug guardados en {PROCESSED_DIR}")
        
        print("\n" + "="*70)
        print("✅ CONSTRUCCIÓN DEL DATASET COMPLETADA EXITOSAMENTE")
        print("="*70 + "\n")
        
    except Exception as e:
        print("\n" + "="*70)
        print(f"❌ ERROR: {str(e)}")
        print("="*70 + "\n")
        raise


if __name__ == "__main__":
    main()

## Dataset Validation & Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the generated dataset
df_hibrido = pd.read_csv('data/processed/financeai_dataset_hibrido.csv')

print(f"Dataset shape: {df_hibrido.shape}")
print(f"\nColumns: {df_hibrido.columns.tolist()}")
print(f"\nFirst 5 rows:")
display(df_hibrido.head())

In [ ]:
# Calculate correlations
corr_score_gasto = df_hibrido['score_crediticio'].corr(df_hibrido['ratio_gasto_ingreso'])
corr_score_ingreso = df_hibrido['score_crediticio'].corr(df_hibrido['ingreso_mensual'])
corr_gasto_ingreso = df_hibrido['ratio_gasto_ingreso'].corr(df_hibrido['ingreso_mensual'])

print(f"Pearson Correlations:")
print(f"  score_crediticio ↔ ratio_gasto_ingreso: {corr_score_gasto:.4f}")
print(f"  score_crediticio ↔ ingreso_mensual: {corr_score_ingreso:.4f}")
print(f"  ratio_gasto_ingreso ↔ ingreso_mensual: {corr_gasto_ingreso:.4f}")

print(f"\n\nNote: Low correlations are expected because:")
print(f"  1. score_crediticio is sampled independently from Capa 2")
print(f"  2. Transaction distribution (Capa 1) is independent from credit profiles")
print(f"  3. This is a hybrid dataset, not a purely correlated one")

In [ ]:
# Visualization 1: Credit Score vs Expense Ratio
plt.figure(figsize=(12, 6))
sns.scatterplot(x='score_crediticio', y='ratio_gasto_ingreso', 
                hue='perfil_financiero', data=df_hibrido, alpha=0.6, s=100)
plt.title('Credit Score vs Expense-to-Income Ratio (colored by Financial Profile)', fontsize=14, fontweight='bold')
plt.xlabel('Credit Score', fontsize=12)
plt.ylabel('Expense-to-Income Ratio', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.3)
plt.legend(title='Financial Profile', loc='best')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Distribution of Expense Ratio by Profile
plt.figure(figsize=(12, 6))
sns.boxplot(x='perfil_financiero', y='ratio_gasto_ingreso', data=df_hibrido, palette='Set2')
plt.title('Distribution of Expense-to-Income Ratio by Financial Profile', fontsize=14, fontweight='bold')
plt.xlabel('Financial Profile', fontsize=12)
plt.ylabel('Expense-to-Income Ratio', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Profile Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
profile_counts = df_hibrido['perfil_financiero'].value_counts()
axes[0].bar(profile_counts.index, profile_counts.values, color=['#2ecc71', '#f39c12', '#e74c3c'][:len(profile_counts)])
axes[0].set_title('Financial Profile Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Users', fontsize=11)
axes[0].set_xlabel('Profile', fontsize=11)
for i, v in enumerate(profile_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
colors = ['#2ecc71', '#f39c12', '#e74c3c'][:len(profile_counts)]
axes[1].pie(profile_counts.values, labels=profile_counts.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[1].set_title('Financial Profile Breakdown', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Visualization 4: Income by Profile
plt.figure(figsize=(12, 6))
df_stats = df_hibrido.groupby('perfil_financiero')['ingreso_mensual'].agg(['mean', 'median', 'std'])
print("\nIncome Statistics by Financial Profile:")
print(df_stats)
print()

df_stats['mean'].plot(kind='bar', yerr=df_stats['std'], color=['#2ecc71', '#f39c12', '#e74c3c'][:len(df_stats)],
                      capsize=5, alpha=0.7, figsize=(10, 6))
plt.title('Average Monthly Income by Financial Profile (with Std Dev)', fontsize=12, fontweight='bold')
plt.xlabel('Financial Profile', fontsize=11)
plt.ylabel('Monthly Income ($)', fontsize=11)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 5: Average Spending by Category
categorias_proyecto = ['Alimentación', 'Transporte', 'Salud', 'Vivienda',
                       'Educación', 'Ocio', 'Servicios', 'Otros']

gasto_por_categoria = df_hibrido[categorias_proyecto].mean()

plt.figure(figsize=(12, 6))
gasto_por_categoria.sort_values(ascending=False).plot(kind='barh', color='steelblue')
plt.title('Average Spending by Category (All Users)', fontsize=12, fontweight='bold')
plt.xlabel('Average Amount ($)', fontsize=11)
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nAverage Spending by Category:")
for cat in gasto_por_categoria.sort_values(ascending=False).index:
    print(f"  {cat}: ${gasto_por_categoria[cat]:.2f}")

In [ ]:
# Data Quality Summary
print("\n" + "="*70)
print("DATA QUALITY SUMMARY")
print("="*70)

print(f"\nTotal Users: {len(df_hibrido)}")
print(f"Total Features: {len(df_hibrido.columns)}")
print(f"\nMissing Values:\n{df_hibrido.isnull().sum().sum()} total nulls")
print(f"\nFinancial Profiles:")
for profile in sorted(df_hibrido['perfil_financiero'].unique()):
    count = (df_hibrido['perfil_financiero'] == profile).sum()
    print(f"  {profile}: {count} users")

print(f"\nExpense-to-Income Ratio Stats:")
print(f"  Mean: {df_hibrido['ratio_gasto_ingreso'].mean():.4f}")
print(f"  Median: {df_hibrido['ratio_gasto_ingreso'].median():.4f}")
print(f"  Std Dev: {df_hibrido['ratio_gasto_ingreso'].std():.4f}")
print(f"  Min: {df_hibrido['ratio_gasto_ingreso'].min():.4f}")
print(f"  Max: {df_hibrido['ratio_gasto_ingreso'].max():.4f}")

print(f"\nMonthly Income Stats:")
print(f"  Mean: ${df_hibrido['ingreso_mensual'].mean():.2f}")
print(f"  Median: ${df_hibrido['ingreso_mensual'].median():.2f}")
print(f"  Std Dev: ${df_hibrido['ingreso_mensual'].std():.2f}")

print(f"\nDebt-to-Income Ratio Stats:")
print(f"  Mean: {df_hibrido['nivel_endeudamiento'].mean():.4f}")
print(f"  Median: {df_hibrido['nivel_endeudamiento'].median():.4f}")
print(f"  Max: {df_hibrido['nivel_endeudamiento'].max():.4f}")

print("\n" + "="*70)